In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from vitpol import ViT
import time 
import torch.nn.functional as F

def print_p_values(model):
    print("\n=== p VALUES ===")
    for name, module in model.named_modules():
        if hasattr(module, "p_raw"):
            p = 1 + F.softplus(module.p_raw)
            print(f"{name}: p = {p.item():.4f}")

def main():
    model = ViT(img_size=32, patch_size=4, in_channels=3,dim=384,depth=7,heads=6, num_classes=10)
    
    batch_size = 64

    # Загрузка данных
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandAugment(num_ops=2, magnitude=9),  # AutoAugment-like
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])

    train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform_train)
    test_dataset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    def train_epoch(model, loader, optimizer, criterion, device):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), correct / total

    def test_epoch(model, loader, criterion, device):
        model.eval()
        total_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                
                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), correct / total

    # Настройки цикла
    epochs = 50
    test_accs = []
    epoch_times = []  # Список для хранения времени каждой эпохи
    best_test_loss = float('inf')

    # === EARLY STOPPING ===
    patience = 7
    epochs_no_improve = 0

    log_file = open("training_log.txt", "w", encoding="utf-8")

    print(f"Starting training on {device}...")

    for epoch in range(epochs):
        start_time = time.time()  # Засекаем время начала эпохи
        
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc = test_epoch(model, test_loader, criterion, device)
        scheduler.step()
        
        end_time = time.time()  # Засекаем время окончания
        epoch_duration = end_time - start_time
        epoch_times.append(epoch_duration)
        
        test_accs.append(test_acc)

        # Подготовка строки лога
        log_str = (f"Epoch {epoch + 1}/{epochs} | "
                   f"Time: {epoch_duration:.2f}s | "
                   f"Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f} | "
                   f"Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}\n")
        
        print(log_str, end="")
        log_file.write(log_str)
        log_file.flush()

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_vit_model.pth')
            print(f"---> New best model saved! Loss: {test_loss:.4f}")
        else:
            epochs_no_improve += 1
            print(f"No improvement: {epochs_no_improve}/{patience}")

        # === EARLY STOPPING ===
        if epochs_no_improve >= patience:
            print(f"\n⛔ Early stopping at epoch {epoch+1}")
            break

        # if (epoch + 1) % 10 == 0:
            # torch.save(model.state_dict(), f'vit_epoch_{epoch+1}.pth')

    # Итоговая статистика
    avg_time = sum(epoch_times) / len(epoch_times)
    total_time = sum(epoch_times)
    
    summary_str = (f"\n{'='*30}\n"
                   f"Training Complete!\n"
                   f"Total Time: {total_time:.2f}s ({total_time/60:.2f} min)\n"
                   f"Average Time per Epoch: {avg_time:.2f}s\n"
                   f"Best Test Accuracy: {max(test_accs):.4f}\n"
                   f"{'='*30}")
    
    print(summary_str)
    log_file.write(summary_str + "\n")
    log_file.close()

    print("\n🔥 FINAL p VALUES:")
    print_p_values(model)

if __name__ == "__main__":
    main()

Files already downloaded and verified
Files already downloaded and verified
Starting training on cuda...
Epoch 1/50 | Time: 68.00s | Train Acc: 0.3071, Test Acc: 0.4188 | Train Loss: 1.8863, Test Loss: 1.6006
---> New best model saved! Loss: 1.6006
Epoch 2/50 | Time: 66.24s | Train Acc: 0.3900, Test Acc: 0.4700 | Train Loss: 1.6718, Test Loss: 1.4727
---> New best model saved! Loss: 1.4727
Epoch 3/50 | Time: 65.36s | Train Acc: 0.4265, Test Acc: 0.4813 | Train Loss: 1.5752, Test Loss: 1.4621
---> New best model saved! Loss: 1.4621
Epoch 4/50 | Time: 67.56s | Train Acc: 0.4551, Test Acc: 0.5543 | Train Loss: 1.5029, Test Loss: 1.2377
---> New best model saved! Loss: 1.2377
Epoch 5/50 | Time: 65.68s | Train Acc: 0.4840, Test Acc: 0.5647 | Train Loss: 1.4392, Test Loss: 1.2145
---> New best model saved! Loss: 1.2145
Epoch 6/50 | Time: 64.99s | Train Acc: 0.5045, Test Acc: 0.5767 | Train Loss: 1.3816, Test Loss: 1.1838
---> New best model saved! Loss: 1.1838
Epoch 7/50 | Time: 65.17s | Tra